# LINDSLY MUYONDA
## DATA PIPELINE AND AUTOMATION
## WEEK 7  ANALYSTLAB AFRICA BATCH B

In [3]:

import os
import sqlite3
from datetime import datetime, timezone

import pandas as pd
import requests

API_KEY = os.getenv("OPENWEATHER_API_KEY", "a6e1608557116c73ec75aa24a5951520")

CITIES = ["Harare", "Lagos", "Nairobi", "Johannesburg", "Accra"]

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

RAW_CSV = "raw_weather_data.csv"
CLEAN_CSV = "clean_weather_data.csv"
DB_FILE = "weather_data.db"
TABLE_NAME = "weather"

# TASK 1: EXTRACT

def extract_weather_data(cities: list[str], api_key: str) -> list[dict]:
    """Pull current weather for each city from the OpenWeather API."""
    records = []

    for city in cities:
        params = {"q": city, "appid": api_key, "units": "metric"}
        response = requests.get(BASE_URL, params=params)

        if response.status_code != 200:
            print(f"  Failed to fetch data for {city}: {response.status_code} - {response.text}")
            continue

        data = response.json()
        records.append(
            {
                "city": data.get("name"),
                "temperature_c": data["main"].get("temp"),
                "feels_like_c": data["main"].get("feels_like"),
                "humidity_pct": data["main"].get("humidity"),
                "weather_condition": data["weather"][0].get("main"),
                "weather_description": data["weather"][0].get("description"),
                "wind_speed_mps": data["wind"].get("speed"),
                "datetime_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
            }
        )
        print(f" Extracted data for {city}")

    return records

# TASK 2: TRANSFORM

def transform_weather_data(records: list[dict]) -> pd.DataFrame:
    """Clean and structure the raw records into an analysis-ready DataFrame."""
    df = pd.DataFrame(records)

    if df.empty:
        raise ValueError("No records to transform — check your API key / city list.")

    # Ensure correct dtypes
    df["temperature_c"] = df["temperature_c"].astype(float)
    df["feels_like_c"] = df["feels_like_c"].astype(float)
    df["humidity_pct"] = df["humidity_pct"].astype(int)
    df["wind_speed_mps"] = df["wind_speed_mps"].astype(float)
    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"])

    # Nice-to-have derived columns
    df["temperature_f"] = (df["temperature_c"] * 9 / 5) + 32
    df["wind_speed_kmh"] = df["wind_speed_mps"] * 3.6

    # Tidy column order
    df = df[
        [
            "city",
            "datetime_utc",
            "temperature_c",
            "temperature_f",
            "feels_like_c",
            "humidity_pct",
            "weather_condition",
            "weather_description",
            "wind_speed_mps",
            "wind_speed_kmh",
        ]
    ]

    df = df.sort_values("city").reset_index(drop=True)
    return df

# TASK 3: LOAD

def load_to_csv(df: pd.DataFrame, path: str) -> None:
    df.to_csv(path, index=False)
    print(f"Saved cleaned dataset to {path}")


def load_to_sqlite(df: pd.DataFrame, db_path: str, table_name: str) -> None:
    conn = sqlite3.connect(db_path)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    print(f"Saved cleaned dataset to {db_path} (table: {table_name})")

# TASK 4: BASIC ANALYSIS

def analyze(df: pd.DataFrame) -> str:
    hottest = df.loc[df["temperature_c"].idxmax()]
    coldest = df.loc[df["temperature_c"].idxmin()]
    most_humid = df.loc[df["humidity_pct"].idxmax()]
    windiest = df.loc[df["wind_speed_kmh"].idxmax()]

    summary = f"""
WEATHER ANALYSIS SUMMARY
=========================
Cities analyzed: {', '.join(df['city'])}

  Hottest city:   {hottest['city']} ({hottest['temperature_c']:.1f}°C)
  Coldest city:   {coldest['city']} ({coldest['temperature_c']:.1f}°C)
 Most humid:      {most_humid['city']} ({most_humid['humidity_pct']}%)
 Windiest city:   {windiest['city']} ({windiest['wind_speed_kmh']:.1f} km/h)

Weather conditions observed: {df['weather_condition'].value_counts().to_dict()}

Average temperature across all cities: {df['temperature_c'].mean():.1f}°C
Average humidity across all cities: {df['humidity_pct'].mean():.1f}%
"""
    print(summary)
    return summary

# MAIN PIPELINE

def run_pipeline():
    print(" Starting Weather ETL Pipeline...\n")

    if API_KEY == "PASTE_YOUR_API_KEY_HERE":
        raise SystemExit(
            " No API key set. Set OPENWEATHER_API_KEY as an env var or paste it into the script."
        )

    # Extraction
    raw_records = extract_weather_data(CITIES, API_KEY)
    pd.DataFrame(raw_records).to_csv(RAW_CSV, index=False)
    print(f"\n Saved raw data to {RAW_CSV}\n")

    # Transforming
    clean_df = transform_weather_data(raw_records)
    print("\n Transformed dataset preview:")
    print(clean_df.head())

    # Loading
    load_to_csv(clean_df, CLEAN_CSV)
    load_to_sqlite(clean_df, DB_FILE, TABLE_NAME)

    # Analyzing
    analyze(clean_df)

    print("\n Pipeline completed successfully.")


if __name__ == "__main__":
    run_pipeline()

 Starting Weather ETL Pipeline...

 Extracted data for Harare
 Extracted data for Lagos
 Extracted data for Nairobi
 Extracted data for Johannesburg
 Extracted data for Accra

 Saved raw data to raw_weather_data.csv


 Transformed dataset preview:
           city        datetime_utc  temperature_c  temperature_f  \
0         Accra 2026-07-17 17:25:09          25.16         77.288   
1        Harare 2026-07-17 17:24:22          15.46         59.828   
2  Johannesburg 2026-07-17 17:24:34          12.69         54.842   
3         Lagos 2026-07-17 17:24:27          23.96         75.128   
4       Nairobi 2026-07-17 17:24:31          18.49         65.282   

   feels_like_c  humidity_pct weather_condition weather_description  \
0         25.79            79            Clouds     overcast clouds   
1         14.31            48            Clouds       broken clouds   
2         11.14            43             Clear           clear sky   
3         24.76            90            Clouds     o